# Disambiguation endpoint smoke test

This notebook runs the full flow using the API endpoints:
1) document extraction
2) NER prediction per paragraph
3) disambiguation


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import requests


In [ ]:
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
DATA_ROOT = Path(
    os.getenv(
        "DISAMBIGUATION_DATA_ROOT",
        "../../../resources/data/restricted/disambiguation-eval/files",
    )
)
DOC_EXTENSIONS = {".pdf", ".docx"}

LIMIT = int(os.getenv("DISAMBIGUATION_DOC_LIMIT", "0"))  # 0 = no limit
TARGET_LABELS = os.getenv("DISAMBIGUATION_TARGET_LABELS", "PER")
TARGET_LABELS = [label.strip() for label in TARGET_LABELS.split(",") if label.strip()]

FUZZY_THRESHOLD = int(os.getenv("DISAMBIGUATION_THRESHOLD", "70"))
FUZZY_SCORER = os.getenv("DISAMBIGUATION_SCORER", "token_set_ratio")
FUZZY_PROCESSOR = os.getenv("DISAMBIGUATION_PROCESSOR", "light_normalizer")

print(f"API: {API_BASE_URL}")
print(f"Data root: {DATA_ROOT}")
print(f"Target labels: {TARGET_LABELS or 'ALL'}")
print(
    f"Fuzzy params: scorer={FUZZY_SCORER}, threshold={FUZZY_THRESHOLD}, processor={FUZZY_PROCESSOR}"
)


In [ ]:
from aymurai.experiments.entity_disambiguation.runner import (
    call_extraction_api as extract_document,
)

def discover_documents(root: Path, extensions: set[str]) -> list[Path]:
    extensions = {ext.lower() for ext in extensions}
    return sorted(
        path
        for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in extensions
    )


def predict_paragraphs(paragraphs: list[str]) -> list[dict]:
    endpoint = f"{API_BASE_URL}/anonymizer/predict"
    predictions = []
    for paragraph in paragraphs:
        response = requests.post(endpoint, json={"text": paragraph}, timeout=60)
        response.raise_for_status()
        predictions.append(response.json())
    return predictions


def disambiguate(predictions: list[dict]) -> dict:
    endpoint = f"{API_BASE_URL}/anonymizer/disambiguate"
    params = {
        "threshold": FUZZY_THRESHOLD,
        "scorer": FUZZY_SCORER,
        "processor": FUZZY_PROCESSOR,
        "target_labels": TARGET_LABELS,
    }
    if TARGET_LABELS:
        params["target_labels"] = TARGET_LABELS
    response = requests.post(endpoint, params=params, json=predictions, timeout=120)
    response.raise_for_status()
    return response.json()


In [ ]:
documents = discover_documents(DATA_ROOT, DOC_EXTENSIONS)
if LIMIT > 0:
    documents = documents[:LIMIT]

print(f"Found {len(documents)} documents")
documents[:5]


In [ ]:
for doc_path in documents:
    print(f"\n=== {doc_path.name} ===")
    session = requests.Session()
    document = extract_document(
        session,
        endpoint=f"{API_BASE_URL}/misc/document-extract",
        file_path=doc_path,
        timeout_s=300,
    )
    paragraphs = document["detail"]["document"]
    print(f"Paragraphs: {len(paragraphs)}")

    predictions = predict_paragraphs(paragraphs)
    print(f"Predictions: {len(predictions)}")

    disambiguated = disambiguate(predictions)
    print(f"Canonical entities: {len(disambiguated.get('canonical_entities', []))}")
    print(json.dumps(disambiguated, indent=2, ensure_ascii=False))
